# About this notebook

This notebook demonstrates how to analyze and visualize the distribution of functional groups in molecular datasets using the University of Texas Vista cluster using TACC's Tapis platform and a FlexServ instance. It performs the following key steps:

1.  **Authentication and FlexServ Initialization**: Connects to the UTexas TACC/Tapis platform, submits and monitors a FlexServ job, and loads a specified machine learning model.
2.  **Data Preparation**: Embeds and writes `dkpes_train.csv` and `dkpes_test.csv` datasets directly into the notebook for self-containment.
3.  **Molecular Activity Analysis**: This step is crucial for understanding the molecular properties that contribute to a molecule's activity. It involves:
    *   **Identifying Extremes**: Determining the 10 most active and 10 least active molecules based on their 'Signal-inhibition' values (a measure of biological activity).
    *   **Functional Group Extraction**: Isolating specific functional group counts for these extreme activity molecules.
    *   **Distribution Visualization**: Generating a bar plot to visualize and compare the distribution of these functional groups between the most and least active sets. This helps in identifying structural differences that might explain their varying activity levels.
    *   **Research Motivation**: Researchers perform this analysis to identify molecular substructures or functional groups that are highly correlated with increased or decreased biological activity. This knowledge is invaluable for rational drug design, optimizing lead compounds, and understanding structure-activity relationships (SAR) in medicinal chemistry.
4.  **Results Comparison**: Displays the generated functional group distribution plot alongside a 'gold standard' image for visual comparison.

**Note**: This notebook is designed to be fully self-contained for easy sharing and reproducibility.

## How to Execute This Notebook

To execute this notebook cell by cell, follow these steps:

1.  **Select a Cell**: Click on any code or markdown cell to select it. A border will appear around the selected cell.
2.  **Run the Cell**: You can run the selected cell using one of the following methods:
    *   Click the "Play" button (a triangle icon) that appears on the left side of the cell when you hover over it.
    *   Press `Shift + Enter` on your keyboard.
    *   Go to the "Runtime" menu at the top of the Colab interface and select "Run selected cell".
3.  **Wait for Execution to Complete**: For code cells, you will see an `[*]` next to the cell while it's running. Once execution is complete, a number will appear (e.g., `[1]`, `[2]`), and any output (like printed messages or plots) will be displayed below the cell.
4.  **Proceed to the Next Cell**: After a cell has finished executing, select the next cell in the notebook and repeat step 2.

Continue this process for each cell in the notebook to execute them sequentially.

## Set flexserv variables (NOTE: These values will need user specific settings before running)

In [ ]:
FLEXSERV_APP_ID       = "FlexServ-1.4.0"
FLEXSERV_APP_VERSION  = "1.4.0"
FLEXSERV_EXEC_SYSTEM  = "vista-test-nairr"
FLEXSERV_QUEUE        = "gh-dev"
FLEXSERV_ALLOCATION   = "TACC-ACI"
FLEXSERV_MAX_MINUTES  = 30
PUB_MODEL_HOST        = "/work/projects/aci/cic/apps/flexserv/models"

## Install needed libraries

In [ ]:
!pip install -q tapipy pandas matplotlib seaborn cryptography requests pillow

## Initialization

In [ ]:
from tapipy.tapis import Tapis
import getpass
import time
import re
import requests
import urllib3
import os

os.makedirs("pred_results", exist_ok=True)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

## Get TAPIS credentials

In [ ]:
TAPIS_BASE_URL = "https://public.tapis.io"

# Warning: DO NOT HARDCODE CREDENTIALS BELOW, ALWAYS PROMPT FOR THEM.
# If you hardcode these credentials, you do so at your own risk.
username = input("TACC/TAPIS username: ")
password = getpass.getpass("TACC/TAPIS password: ")

t = Tapis(base_url=TAPIS_BASE_URL, username=username, password=password)
t.get_tokens()
print(f"Authenticated as {username}")

## Create flexserv TAPIS job

In [ ]:
job_name = f"flexserv-{username}-notebook"

job_def = {
    "name": job_name,
    "appId": FLEXSERV_APP_ID,
    "appVersion": FLEXSERV_APP_VERSION,
    "execSystemId": FLEXSERV_EXEC_SYSTEM,
    "execSystemLogicalQueue": FLEXSERV_QUEUE,
    "maxMinutes": FLEXSERV_MAX_MINUTES,
    "parameterSet": {
        "appArgs": [],
        "schedulerOptions": [
            {"name": "TACC Resource Allocation", "arg": f"-A {FLEXSERV_ALLOCATION}"},
        ],
        "envVariables": [
            {"key": "PUB_MODEL_HOST", "value": PUB_MODEL_HOST},
        ],
    },
}

job = t.jobs.submitJob(**job_def)
job_uuid = job.uuid
print(f"Submitted job: {job_uuid}")


## Create input files from encrypted strings

In [ ]:
import os
from cryptography.fernet import Fernet
import base64

os.makedirs("./data/dkpes", exist_ok=True)

# Data is encrypted here to avoid being indexed by a training run by future LLMs. If an LLM had knowledge of the
# structure of a benchmark training example, it could contaminte the results. The training data is decrypted from these
# strings and written to files. When dealing with your own data, this step will not be necessary.
encryption_key_b64 = "ahz_2H0ho08SvClf-tB8ZwfrWXc0bwuTdxFpsGBIR24="
encrypted_train_csv_b64 = "gAAAAABqaQ3EHqc_ksBPFUGCUTKkt558C5ddwbc5KZlCoIOYy63uMCIjFMkHTyvO-vgrtjUFiVSQquRrdC-68rbd4rpF4Ocu4ch4uxNkiJXd6HM8_7vbTpWSTA2lBaMIHKa8ZBN4T3QovsGP2nL0eWC7q5ZLD7czMf5uMzw75ioUT_zrfud-i6IDhCcZMrhza07XW7sXSZgR95LlWbs2Zo8XCpg0LjNo12dj9jdRjqB3qTfxzG_OsJppuzhUwX_xsuZ1V_wf7b0MMCD6ZQhvJEfBJ41UfrKhg9c0O_0t7PXDk08ESiBHDhXobX-kyKYM7dtp2LnonSmlSBAA92QKJQ6Na-JphLc-DJ12NmYDOTYSY3tDiON4OnAl-Wh-A4OUQMRoaeEk-aXDfaOj4X4RT97uqqd1pwVZBD2Jl04LDC_P-H13bfzIYKvuzcFJLvVQrRmT4QeW00UMrdlGDCAwHxRuXQ2D9WWByo9bQiszGNOtipGxaIc7W60p0RO8AQd9L49gIXCSlLNr05-KgB-ZQnAS5cEd0WzZY2Zi3oF1pPU92YgBBYy3Qm9Ktr5JVCISqLrYxXcSnRgwi72y_3nZpFpSPDY27a0R9yPKuYU9sNnQKUgsufU3S71xZgwTFNmxJ3g3GoAHgcmectEWcSZoIehKEi_gOZn-Xa7zDH0RrGoqVpYfSqO6Qx7qhIOaBNfgfnUVTgIKYpexm1soPkd47R6-RGBe4qcDcB8ECMwTEUESP4qpoFTCEtzZYY-gjwJYTO3Gc9ot10adwmzei9T7q6okew_QZp9FP-NhWCKscoa_3H0kg-dJinOsbxd2Ufn9lmU29ExsJtKgYUIXJWyTbjVX_IYNC8cmUapw0CRJ3y5x6b8EsDIuECfpDKut0AJt_u9JgSHT1QhwxIHFS9aaH7vdAlIjjkCDH0Wo3XP786jaQ8cGucabJVYVuvWEyR6sv2BcnZpFFAVjQQ62NXHBYoBNENQkZlrLrs1RDljNcfWvGc7EH7WGuhSh35hd6Cd5eEDXrI7RZjezsVAgo--hd1z1dpNVGhiSSOiH-YelvAjAMiA8v60v2UgN2gzqYZtRfkdmLr3Uhtp5ZX7uyssQYPmj5VW9IEA9owdvgoF2Qfhw2RG2H01SNRPoCbXJUqRiRJhPA_CfMeW39TpvqiObnWxOzHB0HXfrKM1JdcH0-El34LyQCJIuAhPnVh6De3SrRn3YvFHdkwQkgkcRqLpxkqD03-7HhOnyfexLO1ytxxK1DLyR3rR6NINL5nqSYcuPfLtqRwaPCGGb7yAiAogxk7zhmtGtDhhLDaw9Z4Fhz4RsV-sgwl-h-oMpMEjKlYx5bss7Zh50zwKNVTLLuD-J6VxPhTTLqOf9wMoGwYZlyJrjsQ-SA1rFHPFlEClUBTNH-JWudaBEwDot70lpl2hmpH2NZNk4sJpOcl10AnxlcUVEcYGCVsEQxJ9KtauWd46E22Ig-GdwfbhPHEThxgFWOGT-MLKigETl64elYvHLTEAb2sX_rdwUOJZgWD9lEcOifuCtCeJ1gahqQc5HNdWvcA5WInwPVg5jJrvwBidVhyvWkURDHJ1RTsMftMxilbNOz1Owh9fxk0Nj-fOwfOfWFcaW3Jl5jmS2i0IQ1Nsh70apzuX3aoFnYU9IanUNjozLhy5soB1aEkOfIQzEh0o0H0780vy-b-WDpTwLMxRT_zdPCygqGXDOTHdtkp50NmDPlusvB9fQcAXih_UoTnUZ--YzGU-eNNrHA9sXjO_Sh9c_wcvd5qaPxbTgc7k6N_BvwEcLDRequDqid4JudQyunxx-yC08xf8-rmo5CdiyClDHd9AcKjVIxsIAIeGU4rniGVHpIzWMnF7WsChQo6_JZ-gnIUbnGB7_wQGwfMWfaA5Nv8zWxvSoUaAFDjYLhyHzmLY3C6Mv17zVRJRgjpLqVyY7hEwZhfYvTbI8zYOj5TJhbaHb8fq5n-sFPlYkzFS2Ued6mOKm8E37u62oXDNkGO9R985NMsietnA7vAAaKj6gXWiSI3jRnlPssOWQEWVyYo1BC-zH5vt8oY_32h1gZsOK6xX0vGCDJSi51EqQekX3S7LzSU-bXZDEn2VstpXJikDFF5_virouOU-Hwv3nWA8VA5xXjGIy95awT6LaxK1dE4Ckdy8GK9SLohDpwq0-QntmMxp0DqsNjVwRWgpX0GYi8aowUiJgg0uAxE8hVT7KEujPnASRJBFGibjoTQMTl3-mQkLU5jiYd6jxLQmbHHwatwVzb3t4UUAFqydxU64iwF9rwM-MvQ1v7vlHyDyo9WgW2PHQyEHE1UBey6nblrXFyC5XlG7KCdqzIWvTSeN-kYELpiaq3CED9SjS743n6dGdIGJrGFeoAsduSi9wDe6rUakPXYN45KTvzYmFsvpEq3438oRnHNjG0ac_LxPvXYdpXZLGRPt8YgJVc_HWRnmcir4y47vrYouQo-LTpWPE6U01-LVe5bnBvj29gqRh-jZ5qgRajiYXRETizUawvkxnnPt-2T4UmNVN7-tjwNNNR9bmqF_eRFeZ-nlCc25bgpQIT05JnQz4D21ahtZFlNby4vD8JgDiOlyybX_pqBLOh1wNI3mw1sph4NUe4vBpCHAVT2EC3rJQfOuzy8xpMKQJzkmA0X40nVe04B1F5nIdhLDKzwC3Q72Y5nXJQ05lypRJj2LuHxVvEwKGBlvqORM3Qh8Hr76ZAQUVPT-Gb_Xf5WQv4bzBZVgysZbVBxGgTJHPKZwCdbklpZU3PhNWTm7J4n_Z22hmfi4D2h1WENtTaBKkK_-_cHLBnPL10_VosJ7heFprx2IiF4DWubVI5N2dGVA0Zo1llplch91RsDpNN6O9b6STeEE-hv4xqPp1sBMBYxJyioHeTyViCHmzt0fa6qKMBLS8Dq7AAsY6wLyk12Yh6uXw4BNaeNFa5WT1mlfWzb-eeEmQJONOFTQS-OKnBh073dUDVDGTfuw94ebG_N3UYTh9WH5H9SUF2BymBXFn5zPMNj2EmnROlqjNC---lvLRlskNt6nP15O7vOJpVJvpd-RpxBEebDY2loqXitMnOjs3a8bmJy2jjTNN8QdhMZAwv9yPBoeRh1wm9jEszMztUJ3feYc_y2z1M8LCMlSj0IOgWgAwy9UPGf5KKCiH8zz21-ZRs7ByAtNX84smmWjRSpIv2Cu7FawenjkSv75JX7NH7SEGdozftcL2OViBQxRG-W8YzagCJflFz6vb8yrk3pNaWOJWrpzIYNnphV1fMVBsMftb2axRochkZTznrOgcgOoL2Um1AQSMccrEpfxypfNRYSWUTLLvhV6oU93l-9uREPvzmqxZFY6q_b5Oj-LEuzmLMUIqJxGFusx9_mvcovDTKk5q4houXQgopuFj9y4YeY344-6_YLRKGmTVP7lUzc-Gb2qLbnCOMP0bu8vhWHAkAk0GL--VGyLzpsejYAVEs5LfjQyWwp7JHP7P7v-uDKx-06niR-lXktq994HWaAupb8O_eKsI9xYn6njU_8lS-59RJ0n4Bk6onYWuwQLD1X4Nspy2Tvx3ltKWMy20HergOSj4embUqsn9mPD_aikUh5Rz86H27BbkbnOKXzmlKyoP50XadS3WyPZDt0Z5P0I3WVy8cPlZCMsI_dctcl6BWBTD50qZhdttKJWKQ__W0GX4dU_vN5TRZ36oooxjok-zLasI2_IcVXf6wZwAr2xDNpNckymjiqIiVyUp63JYuKzp4dOpaezElHwgJdun3MUETvdHCSskpKBNjG7dFzrMcCtig3GgSI6Cklqh-_Ou_0A9rlzczmTTNIqZVAmq2cxGpJKDymuc-7kU0_Dk50q1Sa8IJFT1MpltE1uShIiJ-j04QBxuZ8FkC3bwky07J9RyOFEPR96dPalEDK-igS91q99WztE9upQoHj5qjJ10JlA8TAbyr-JOqLRupiLplC15TXQBORBFvFUt4Iq0-oJHyZWFi9Dn6to481jMvhRGkNZQ03v9z1QaQcEUyWIBFpUlB3tGOdzd3kO28aeh1IR9TzBCP3NgesQmau-OV4jKM13k5Ji4K3J0qkOPPeJFAdaZ729Pw3GH0HJYkU3Shg4hWPRUBMcwcxBFtrxNDMk6f63B-yOs69TiDkM4JmSv20xaXzfi0-iEvMjbEwfHMXzKtztFxgo8lKocQSUqzyOd_xIAONHM4cjWDRXxVWhVCjS4xAx-jXI1MBlTxs9ESjz9CkvD45qrpgu8IrCLZie4Md-IQXib4XqI9aXamR7BLX1wQQfkggljxBSr3BfUCpAp8WAKaEGe-1w2bR6RPb6xKvYL7X8qf-CZPfVyzKTSiTGAukX1rlSeuYY5Ojhi61t0dguKbtnU2UX-2xdvZ_L5mClKNEyLxjijHVNMwGPXBCndRXZeTiipdTgPJOYSZFjpQS5X3ck_MMOyOWDtrgN3j2vk7iYLnAnGGT-6I_8XtP-WSeFx9XUE--if8VRyZnm3NLIU8hkyMTo73Ta5tYGxegwPbM4DV4BavlRw1Djoad8PgLk20HQ-weEM8jcK91VwH4R8nrvrC1ymZR7pRGpqIRIrpiAodvix3py0yfWoBfhaW51hYwChv_SqTSJG8lsnXy2pNwxdvc_o2IKgi8GPAcWLvzVFaLoSFMuW236S-sjHTXedPisx8nkjMvZqisPEN_8_-KG5yLisi1lYnOeV8EqdaXvesgr1bUhEnFXmp9Q0swaQL2awUOZV1NpeLfyd0vlB9ujJhjwnapHyZXaVoi58FEYgUBr6ZfsQ0LvMhHDRcntGzOuwyNHxuIJkvYxCfXo2onJ4zREjrJ49WzWOqowRw-Qcbu6qgTetEZrh2Mz_LDlxjOll8Oqcp4yg6H5XvVsHwhUJuWHrUk2Ft8GMZD4HxOz7mXlNY_ulxoCKN2oTtDTAyq5_o5Kh9f8s_GwSAqLY6-Y3BfPdV8pZXr_EYJ5Mb310vdlsR-6414ARTymTNa0p-i7fpa32X4Vu9O5q90gIotheR_wrHXpw1M08Iue3cFOMRq7ljy64iTia6mqN7ZX1yuk1ZPfJTeJFMRqRrJV1qA17X63F3jsg-kn0B-l0XilZMubfG5dd6JEt2QjtGlkTUqoZBmuAJT9WJBV-giGyYgZFQR02dubDwBtkVOVlaZVXrbEzDN89R0-U23ZAJ5ox9SsfVVtSYezSEMxE3ZMgKkQSkSowrLYhyuAqUyIjTiNsqQ3GzDxmSroTfKU_MnRAIZ_FlaWxPH5BvkrDA0OKr-DH9drwXxlcoZXry4Ss_agd3UMGpxOkPZnmaFDdh97WqbwVQgQmvHTWKEesiO4PukxRGmZmHSezT5BQ7ap-FpmE0Gay8HJJ10HxH0y6oLQSalfzm0mgiPboZnceNldbsHvd97UQCRcHcnI0n9HRippttri93K6qsYiA-y9CpeVeIXq837BxACbTYdIH473ws-ZwkZ4I5ut5FPjZd6dsslRmYWTa2a4MNGssiSX72o7_l5RiPkbwxI2-WUsClyTe31Pt8mnNk555QnhPzJJhUsFP26vSE0sCxqMHCx5YAM-sSxo3DhKvObXiUuE-nq5Khg-4spvr5JFTkRHmEmgEOFcqtzL0PaAztTqDLBxAlaugBElMnIbSP_ovb7zXM_O_aQp_6umOjnmmcp5UnYuiwUnNjmYVprZ_XYIyAqVLpBcXa_OSp4LNhZ8lI0cdAFKPC2UgxGdcm2nPchsM89U9hzA7lhEYVthS4AL-xbZxLKWUajOcCBX9qV1POC-lCQN92RpCctH3yvcxoA2csfzGvIRrdAJsJIJKrVknZMxsB46cr0hc8_QI-ihC-RIJXaSHT5Xb6hzSz_evCATfN7Yisv249HQhvXoFuNVUIxkzWRAFPCPuajuaFEwvw_af6kOJAcBGwFoKN6ggVRJy9A-WKTWSjgyPQ6vSOneyYe61QznrT-5dCYj3qeCTwIRoQinyjRSNYgL01SeVKpNvG3yxEs9Wz_Na_JoGwlsgVL-AcUOzfDHnejx1l6YujYoZ0gvqXkAW97t77nP0fB7BqosIXAqY0pPMs73A6f_R5sEwBQAA9KJkARN7nQ3uE2OktYq4f8nmWQ-mcMJtZpYH9vIkQmiY9liH3lu4hSrzkO6jkXiEKNBkIOlCoTD83GlOcE2haophsRq-RysDEB8YC_wfh2xcL88ceCrSPYa5L0IBro5Go0kOSBZE6jdV2BNr4Nh_FhSp5Zf5wXDqPuJiHDmgpqvEuGQLCyI-0ZTRqLfur70p923APxeJjphyGdRCa7HB80hu5Vy1S8iirIR8-Lw_8pNI6LsZjApTus8IQfcA04WTTXLs6egV4n50RvzU4ZZ-AtNbTG_MsvrmjaauFch8wl1IH0qpWL1nMweJYdw6fOLmW5LU6O-CGMZv7U0S6F2ToTaPHrscVgw-2KtEIfiMQQ3CTDsgx0OH5bmN7f4zISC4b2NquuHkqSDlnWihHkdCDxjDuxZWYM-xIV6PHiHz1Pp-OtFxV_4dEtWwzmdhN3PU77-s_AEUqJmPRsfhGpVkpQx6aw1a-7r6bREzIuG78wBDEIc0KUVE0qiCN5B4aP2gXA6hryfEl3Vuu7-YpBFTkUW_rBoGRbFyHDAdvmSgBdFHWOUAtLgD9rlc5XJMKT1pXvnEQtF7OHl4i40L8X-ZHZdXZQalJLnl1noUcXLyFCY-pIHaZ6IkUk5ndlkR978N_3R0IDVkNOxkO_n4T8iuTS4Quu71eUEbbE8btjwmD880iweD8gPyouaSP3ijRabWVSoqtXv00xZ_udJICzRZaVu9LCPTSYSncbmZ3MxkGAoMND5xzodzZgUNguelBNpljk7PaBK2epPHusuLm_1Rp3CqZjVbzSdJ0rDsh02HET4d8O9DakZQ1YxtwKB0_gOeTZRetGv0yxn5TbPxK9VfLDuPiEFDjQ3JReDtRALogDn1dskcDxeyUuOAKYOaA6KRrjANnHdDhbmeR0aLTC6yg3m_SbaWKGWQYnUlBQte7hxACUiY_dGLmHnUCrwHYaIXYLnmzRB8ftn-sxe00x6FIaa529jFJFnU7PBcMMPxNOgQi7y0eNdw8Kg-Pssg1JiTN1f0s2oY_Ypl1N8g8_-iAIeFaAhA8yvJCzds668I35_W9D9oD__wK2FFl1qoVQ5XgVc57qfeqRS8iat9b4nhaWNcbU3RyWFQlUhrNdTEnkjnrcTO1JpbWTWQrCNU-b29Py0aD0xHczP_Z0IE6z-7ewv7sStzBmXCEcLxG3LKeH8WquwV-qJAaeb-_fFeLo6-gI197A1n8QkV0iJv4-hR_ngVFcjVAtqQCZglTmzQnH1TojGHNIGCJA605wTzypy4Dc-ys2YIIxIedetIftZCYFl-uHemERXhuZM5pHEHgrUyZ2g1t_4qOq-zQj2xONfYX0kXX2fx2WJdq-sroX2UTak_n3LUCuI9Pahg02bKplV4ZLPcAxXfkysDDT47-Ca2SLKrw7UBkGROMN98ignmPaYUplHW-pINHG8LlmztcdxN5gp-P-y2DAy9l-0qCkOipL2OMaj9RG3VUpVkLbEG7wbCpQJnEGzRCjehuu0tzdGppassKtNldDdvsr6_59uLA2qBrHTWYEtyujjn3OhdHSafNWyrrN2nSNLdEMjlZemA5ZyJi7GbMJMPKU7fGbSJoB05KVB35K7DJQQ0XKfZdsB16sWKHJ7GUyZcGusNjDamZKfPeAXBI1swV42Q3aP6d6mvicXUUc0v1_jhIA2via_vWpJTQVB72NsvBfHc_TAqGq0eesLwKczJ-1g_B0eMFESYpY4dhWDtydTfeAKYq2xm8AMTFD_8T2uBhohw7xtwFnKiWC-WSPmYMZzzjE_ImmI9kyvQ-PHiiK-REso3ZvHz5S4qxMH6wH5yxdeDVn1PTDWiwWGHzJl-zPAPDYJ1nZR27cM1VYPoz-4FYks4F2LKhEBr7JI9U5W4o2B6ljUdQIbQW4UU6ME3zIxz2o8JJLhX1Av2m7fCm_eZP228Z7z6G_yvgoqhyuB_oryyOCDIQQsU4TXyZz3Gw7crik6kAwhRW1zzpbrgqYWSz7DZuoQWG9bfP_xMNUHrY8ziVGhu8yJ0irM19xqxS2lj9PtvuyXHIzSHrIQ-YmsCEP_eI4BwCLI_R80IpPupwt5KPC2ZWGjRt1jgVOL6QRI40uVE_9Mc-rTZ9U-czy8iflDdhS3bP9TtOOP5jMd5LSg1CxaHdJBWOEwrHzcmRwWQiuJy8yVHYMVVJ27xZHv2jzbTlIXhVL8gM-KNHMmzL8UQlQ5Vd6Chs8ef7J1JfdESvsgQe-L4bzIKJWIJiewmFg9SKCffoSUJQ9XWP4EExogrpv--arCFkXfJpZxElIuMhxZNKcSu43MN2fcnpWamcY22reYDYOdogQUHH4s9ZPAqDvBgfW7KG2dLy8ci42-60qLqH9wdbkrZq1mz6JjnbHfCq3fGjSOLnokzbV-ixKLTaIjPS34UX45oCtqZFTinEA1YncTzhQBCC_ugmfDKavTcYsD6G4mczfjZuHv0Rwl7OnDWF4vrVV3OSZP4aMi5xCawQnMcc70o3m6YyyUuV3yRwWR5LcilOHssjaYTFfCitEvWbDZ3U0488FfWmJ9rzpwmfk-Sa_vT6Wy7Vg4xIYFBJqKjGfT74g45_rKjkvbJAzOdXMOXmGE4lIV-wuo1ZNwQOPX2czjF-nxjcSmoO8ZQAW2RSiJfKKoKkLDv82peVCDiOw0roUxir3Uod7RMopQ="
encrypted_test_csv_b64 = "gAAAAABqaQ3E2DDuUVvO0iqmPWRGwn7a-oDO5MSdZ3eC8R3ZST5Kdb6StTP7Tzu2Xtn5S5hrXTDRdLgPZtOeVoOycDeRXNNA_so4Hisqu2BkMoM5DuYudxpZnH-dAxYJIr1Kh84fgyHv_JK6x419VDJx0CNhQARABEeyhPW3FVXTxyKmbdPl52-3c4C2G65UVeaZs7o6SyWBtBhlKlgC1O34gS6tsp8iNO8_54Kxj8Qo2krcunZdcSz-1Ot6azgml5znkJmOBDN0aq2KcZ99lNVV5ghKqX-5uj1TQyrx2kdh77KyHKLSN5YmbyyV6A4rJiQ3cz1hPkIARgzpYhxKN6ejRQ8n23JMwxF8PwxC6tIzaSr_qEPK31-hvkE6tfCHLRpolIUKzwlXNbmxvNDSQtS_UpWFnifWpQMevuEsIPCRVNdkqvsiVuekuxKppyajvIQDFBW7GXQasFCvTo869SRA0NPX6OSjs-yEHskizy2ZF2Z2fsX8i14r5lGEdy5BW4LFyAXxojEMaov_XYKJAZmSm8THLRRZJJ0_uThgikue1dSl4ibaaRd1bmnprgK5h08VmtuYBp_Xbc8OebkoXMwOJC5Eexh--TJoPwbn-rLQVWqkohWFYjjhvpJyr1ll77PPCFsU-amLSkoPVFrW_YeE7Ps7V130sgDgodulgXndSSTQnrvTb8FRL-fmLLeriKqbIGKInMcE6hP0i_q3WFBBBdBudNlamNarR1x289dyzh_4ndVJnhK115IKFPdFR8MNGqomPirHcCy6ldYfG5rR-pW9HumuBifiUUc8i0osJeKUpnjpUbQ7aQtxBSFUE0JesS_bLxEJSeQeONYfFwABYWVjmgmMxXzJ47-jvHHkoiCUYK5alW7hud6XzgCh6FpJQtiG1w10fmNkwiVJxXYhlYytXFJHJfwGxGfe0iUOOFhnibOQMGkLrOQQowiPF0A9euU88Yg67lg8T_fHq53sciyRK6QbzZ_oDwljEvixX_dFS5w1Hp0ZH4jMQrErkWce4HWJCYmXEyMlfIH4H4iQk27Nx72DmEIBgUa9VCNtpIYBQ75zLV-JrgwOyTWXWrhEHx2TxFKR-_stBMhvKM5eDAkT4lsDaHbwASYD9r0d-iq9uWU48xZ4hKWbAGHiByL9c3PqBh-9QZVul4DqFTaBobK2i28SzS5rnQlGHVAzfXSqEIWvAJkHYCxve_jLyN9PNTMri_MYm9q2DbYC5tTZ1BvT8PQtmu7usHhXF8cmyQmPqfgYrMqrV6a72yQTxiINDzSMe9lvAFjj_ioI54cq1R1hm2V3PRKZNit8OCpJxUJiABWDnqs8EzjR5-8gUPYzMLDWNB6NITyP2VpNfUkuWJ8DgnTW5qKiOtupdbpQnObq-M23tjahmNhMAASzodNaZO3pwKt6WzbeZELYtc152YR5Ic3CqLf4DHKkGk9gZyGXqOTkeEfD_23vw_fCqqWoIK4ZE-0W-UgzdJYQPPcZA9zfCNPVqj13NZ9wkZ9rqKT06q5goWB9DoGJ1_7CIDZTGGl5g_JttZarPquZKeSsmdkfftFwdWs9GpbQCPUrRxBUKB48Uytw4cTUELw8w-Wmhz2z1ERmLFbVWohp5e25ND-eYfkAJir_iPcDk3NpIimKiYWI6NLgasVN_HpMmmjmk6eN1WYPcWoA0kVa3B1uwpppaMNIaGusQ2rqRKx_5H5BG7ZRJV-OnYNcYTYUMQezWkncKQLvodm68AALprE-woCAUGAz4Hq-jxvqO357e-1iy0Sf4TpP-IS16wdCNqyOkK5EMPp9vV2Hupm1jggPlQX7BpPqgKCp-2-mMh9AscN_vw-w2tVG35tlaXpHNlPLfpIJDopEWBP9wLD-2X-D9O5kMTOEQhBPCaNd-K0APCNqxkuU_kO5USQoZokEfuIkmDqFsdwLE6QzemC1oGVUzo0y_XXkA4mFRUUhP6Dqytx3frpHFfx-8sgSaydY2zNJi9qLvKIDEjxy2NlDX8Bc_nnzU8lB_-Cj1h7rIVrejBQ427gf7l7by0tZCN0pnMCvzmkHdhi66psX723IC3u84tMu-8_A8Z1CvtFSYZiRdhMV1iijLwNL9W76afEBtBuJcE7mHe0Q_Vez5YQ8udaqU1hxXL0BolULldDSWVL_xSn8hrEDCpH91NWe7vuXzA8k85kS2ake7bUBPS4YiHV5s6j2dld1tJb6kaz0vOcAV5A7UW2A0gvjt8PrMg2IhMSeeqVB6st5gHwITWnOMV3hblmOrPcNecjQAAzmQ34H3QLbQ6IzzQbFaXisqzQ1UcSP9YkMKzYCmMZUbtxoPjxLCESQrSGufqITEZ-KEIR1aqIy_ClCpoAxsNpMo09teuoYzDxf5FlxhTyKYw-aGPcSp_V2iltMehNJhdC96S2_xcQNBXbnhZ4G_ZQfYw9tiaIGLc8jHjTvsb84INZDTooZuHYkGR_e0KjI9R-lXs7k0M4qRW9oM2MtGX9amcOuk0Lk1EKjdux0_hP_6XTMMTaOlYB7kO01qZhVjv6uNN-O46FSBAjSnUsXllj9DyWWXicXoTu8hSnEq_V2cmt8RJzzmCubVjtbtNgaYfn4uG-6nSrpNUeGtdkrtivtHg6p-JFbUwBs8PrBvHlkIJ3yylKqUfrQDpJMjMH4DjZJ75VG2hw6bvwbDxGEiggpqjQnZI0hFi8QEPyUj2P_xq14GqdU3rjycRt_2A=="

gold_answer_image_b64 = """...
"""

# Write base64 encoded file to filesystem
with open("dkpes_molecular_activity_analysis_gold.png", "wb") as f:
    f.write(base64.b64decode(gold_answer_image_b64))

print("Created : dkpes_molecular_activity_analysis_gold.png")

# Decode the base64 key and create a Fernet cipher
fernet_cipher = Fernet(encryption_key_b64.encode())

# Decrypt the train CSV data
dkpes_train_csv_raw = fernet_cipher.decrypt(encrypted_train_csv_b64.encode()).decode()

# Decrypt the test CSV data
dkpes_test_csv_raw = fernet_cipher.decrypt(encrypted_test_csv_b64.encode()).decode()

# Write the decrypted content to files
with open("./data/dkpes/dkpes_train.csv", "w") as f:
    f.write(dkpes_train_csv_raw)
with open("./data/dkpes/dkpes_test.csv", "w") as f:
    f.write(dkpes_test_csv_raw)

print("Decrypted DKPES data written to: ", os.listdir("./data/dkpes"))

## Wait for flexserv application to start on TACC Vista cluster

In [ ]:
TERMINAL_STATES = {"FINISHED", "FAILED", "CANCELLED"}

def get_connection_info(job_uuid):
    job = t.jobs.getJob(jobUuid=job_uuid)
    if job.status in TERMINAL_STATES:
        raise RuntimeError(f"Job ended before becoming ready: {job.status}")
    if job.status != "RUNNING":
        return None, None, job.status

    output_dir = "/" + job.execSystemOutputDir.lstrip("/")
    try:
        content = t.files.getContents(systemId=FLEXSERV_EXEC_SYSTEM, path=f"{output_dir}/tapisjob.out")
        text = content.decode() if isinstance(content, bytes) else str(content)
    except Exception:
        return None, None, job.status

    match = re.search(r"FlexServ address:\s*(https://\S+)\s+FlexServ token:\s*(\S+)", text)
    if match:
        return match.group(1).rstrip(".,)"), match.group(2), job.status
    return None, None, job.status


flexserv_url = flexserv_token = None
poll_interval_sec = 5
consecutive_errors = 0

# Loops indefinitely: transient network errors (e.g. RemoteDisconnected from a
# stale pooled connection) are logged and retried rather than killing the loop.
# Only a terminal job state (raised as RuntimeError above) stops it early.
while flexserv_url is None:
    try:
        flexserv_url, flexserv_token, status = get_connection_info(job_uuid)
        consecutive_errors = 0
        print(f"status={status}  ready={flexserv_url is not None}")
    except RuntimeError:
        raise
    except Exception as e:
        consecutive_errors += 1
        print(f"[poll error #{consecutive_errors}] {type(e).__name__}: {e} -- retrying")
    if flexserv_url is None:
        time.sleep(poll_interval_sec)

print(f"FlexServ URL: {flexserv_url}")
print(f"FlexServ Token: {flexserv_token}")


## Check health of TAPIS/flexserv job

In [ ]:
headers = {"Authorization": f"Bearer {flexserv_token}"}

health = requests.get(f"{flexserv_url}/health", headers=headers, verify=False, timeout=10)
print("health:", health.status_code, health.text)

info = requests.get(f"{flexserv_url}/v1/flexserv/info", headers=headers, verify=False, timeout=10)
print("info:", info.json())



## Load the specific LLM we need to use
NOTE: this could take several minutes, we are loading a 32 billion parameter LLM

In [ ]:
model_id = "Qwen/Qwen2.5-Coder-32B-Instruct"

resp = requests.post(
    f"{flexserv_url}/load_model",
    headers={**headers, "Content-Type": "application/json"},
    json={"model": f"FLEX:PRI:{model_id}"},
    verify=False,
    timeout=120,
)

print(resp.status_code, resp.text)
print("done")

## Construct first part of the LLM prompt

In [ ]:
task_inst = (
    'Visualize the distribution of functional groups for the 10 most and 10 least active '
    'molecules in the DKPES dataset. Save the figure as '
    '"pred_results/dkpes_molecular_activity_analysis_pred.png".'
)

dataset_folder_tree = "|-- dkpes/\n|---- dkpes_test.csv\n|---- dkpes_train.csv"

dataset_preview = (
    "[START Preview of dkpes/dkpes_train.csv]\n"
    "index,Signal-inhibition,3-Keto,3-Hydroxy,12-Keto,12-Hydroxy,19-Methyl,18-Methyl,"
    "Sulfate-Ester,Sulfate-Oxygens,C4-C5-DB,C6-C7-DB,Sulfur,ShapeQuery,TanimotoCombo,"
    "ShapeTanimoto,ColorTanimoto,FitTverskyCombo,FitTversky,FitColorTversky,RefTverskyCombo,"
    "RefTversky,RefColorTversky,ScaledColor,ComboScore,ColorScore,Overlap\n"
    "ZINC04026280,0.24,0,0,0,0,0,1,0,0,0,0,0,DKPES_CSD_MMMF_1_32,1.184,0.708,0.476,1.692,"
    "0.886,0.806,1.316,0.779,0.537,0.528,1.235,-5.804,1045.931\n"
    "ZINC78224296,0.278,0,0,0,0,0,1,0,3,0,0,1,DKPES_CSD_MMMF_1_31,1.063,0.765,0.298,1.346,"
    "0.904,0.442,1.31,0.832,0.478,0.48,1.245,-5.278,1122.302\n"
    "ZINC01532179,0.686,0,0,0,0,0,0,1,3,0,0,1,DKPES_CSD_MMMF_1_16,0.965,0.633,0.332,1.896,"
    "1.143,0.752,0.959,0.586,0.373,0.363,0.995,-3.988,770.823\n"
    "...\n"
    "[END Preview of dkpes/dkpes_train.csv]"
)


## Complete LLM prompt

In [ ]:
FENCE = "```"

SYSTEM_PROMPT = f"""You are an expert Python programming assistant that helps scientist users to write high-quality code to solve their tasks.
Given a user request, you are expected to write a complete program that accomplishes the requested task and save any outputs in the correct format.
Please wrap your program in a code block that specifies the script type, python. For example:
{FENCE}python
print("Hello World!")
{FENCE}"""

FORMAT_PROMPT = """Please keep your response concise and do not use a code block if it's not intended to be executed.
Please do not suggest a few line changes, incomplete program outline, or partial code that requires the user to modify.
Please do not use any interactive Python commands in your program, such as `!pip install numpy`, which will cause execution errors."""

DATA_INFO_PROMPT = f"""You can access the dataset at `{{dataset_path}}`. Here is the directory structure of the dataset:
{FENCE}
{{dataset_folder_tree}}
{FENCE}
Here are some helpful previews for the dataset file(s):
{{dataset_preview}}"""

prompt = (
    SYSTEM_PROMPT + "\n\n" + FORMAT_PROMPT + "\n\n"
    + "Here's the user request you need to work on:\n" + task_inst + "\n"
    + DATA_INFO_PROMPT.format(
        dataset_path="./data/",
        dataset_folder_tree=dataset_folder_tree,
        dataset_preview=dataset_preview,
    )
)
print(prompt)


## Your LLM will give you back a python function to analyze your data

In [ ]:
resp = requests.post(
    f"{flexserv_url}/v1/chat/completions",
    headers={**headers, "Content-Type": "application/json"},
    json={
        "model": model_id,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.2,
        "top_p": 0.95,
    },
    verify=False,
    timeout=180,
)
resp.raise_for_status()
data = resp.json()
assistant_output = data["choices"][0]["message"]["content"]
print(assistant_output)

match = re.search(r"```python(.*?)```", assistant_output, re.DOTALL)
code = match.group(1).strip() if match else None
print(code if code else "No fenced python code block found in the response.")

## Run the code just returned from the LLM

In [ ]:
if code is None:
    raise RuntimeError("No code was extracted from the model response; cannot execute.")

exec(code)


## Display your analysis image and the 'real' answer for comparison

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 1, figsize=(12, 9))
axes.imshow(Image.open("pred_results/dkpes_molecular_activity_analysis_pred.png"))
axes.set_title("Generated (yours)")
axes.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 1, figsize=(16, 12))
axes.imshow(Image.open("dkpes_molecular_activity_analysis_gold.png"))
axes.set_title("Gold (paper's original)")
axes.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
t.jobs.cancelJob(jobUuid=job_uuid)
print(f"Tapis job {job_uuid} has been cancelled.")